# Download Wan2.2 TI2V 5B models
Run all cells before starting ComfyUI. Downloads resume after interruption. `HUGGINGFACE_TOKEN` is read only from the environment and is not written into this notebook.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

WORKSPACE = Path(os.environ.get("RUNPOD_VOLUME_PATH", "/workspace"))
COMFYUI_DIR = WORKSPACE / "ComfyUI"
MODEL_ROOT = COMFYUI_DIR / "models"
free_gb = shutil.disk_usage(WORKSPACE).free / 1024**3
print(f"Free disk: {free_gb:.1f} GB")
if free_gb < 25:
    raise RuntimeError("At least 25 GB free disk is required before downloading.")
if not (COMFYUI_DIR / ".git").exists():
    if COMFYUI_DIR.exists() and any(COMFYUI_DIR.iterdir()):
        raise RuntimeError(f"{COMFYUI_DIR} exists but is not a ComfyUI git checkout.")
    subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFYUI_DIR)], check=True)

In [ ]:
FILES = [
    ("diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"),
    ("vae/wan2.2_vae.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors"),
    ("text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
]

token = os.environ.get("HUGGINGFACE_TOKEN")
for relative_path, url in FILES:
    target = MODEL_ROOT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 100 * 1024**2:
        print(f"SKIP {target.name}: already present ({target.stat().st_size / 1024**3:.2f} GB)")
        continue
    command = ["curl", "-L", "--fail", "--retry", "5", "--retry-all-errors", "-C", "-", "-o", str(target)]
    if token:
        command.extend(["-H", f"Authorization: Bearer {token}"])
    command.append(url)
    print("Downloading", target.name)
    subprocess.run(command, check=True)
    if target.stat().st_size <= 100 * 1024**2:
        raise RuntimeError(f"Downloaded file is unexpectedly small: {target}")

print("All model files are present in the expected ComfyUI folders.")